# NB07 — Event-Level Rank Tests

Complements NB05's daily-PnL Newey-West and Memmel ΔSharpe tests with
two power-friendlier event-level tests on pooled OOS observations.

**Why this notebook exists.** With ~165 trading days per OOS year, the
daily-PnL tests in NB05 are underpowered: a true ΔSharpe of +0.3
fails to reject H₀ in a single year more than half the time. Pooling
2024 and 2025 events directly recovers ~360 events in one statistic,
trading some autocorrelation robustness for power.

**Two tests:**
1. **Spearman ρ** between predicted P(positive CAR) and realised
   CAR_2_21 (the +2 to +21 PEAD window). Note the model was trained
   on CAR_2_11 — a positive Spearman on the longer window is a
   stricter check that the signal captures genuine drift, not just
   the trained window.
2. **Mann-Whitney U** between long-bucket and short-bucket realised
   CAR distributions, with quintile thresholds frozen on val 2023
   to match NB05's trade rule.

Both are reported one-sided in the direction of the modelling
hypothesis (higher prob → higher CAR).

**Sections**
1. Load model, rebuild feature matrix, attach long-window CAR
2. Freeze quintile thresholds on val 2023
3. Pooled OOS Spearman + Mann-Whitney
4. Per-year breakdown
5. Outputs

## 1. Load model and assemble OOS data

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
import config
from src.llm_agent.data_loader  import load_sessions, load_prices, load_llm_signals
from src.llm_agent.features     import build_feature_matrix
from src.llm_agent.event_tests  import quintile_thresholds, event_level_tests

bundle = joblib.load(config.MODELS_DIR / 'event_study_best.pkl')
pipe, feats = bundle['model'], bundle['features']

# Build full feature matrix (arm C = LLM+Tech)
sessions = load_sessions()
prices   = load_prices()
sigs     = load_llm_signals()
base = build_feature_matrix(sessions, prices, sigs, arm='C')

# Attach realised CAR_2_21 (the long PEAD window — different from
# the training target car_2_11)
car = pd.read_parquet(config.CLEAN_DIR / 'car_labels.parquet')
base = base.merge(
    car[['session_id','car_2_11','car_2_21','sigma_e']],
    on='session_id', how='left',
)
base['y'] = (base['car_2_11'] > 0).astype('Int64')

# Sanity check: did everything align?
print(f'feature matrix: {base.shape}')
print(f'OOS years available:')
print(base[base.fiscal_year.isin([2024,2025])]
      .groupby('fiscal_year').size())

feature matrix: (1496, 31)
OOS years available:
fiscal_year
2024    198
2025    165
dtype: int64


## 2. Freeze quintile thresholds on val 2023

NB05 uses the 20th and 80th percentile of val-2023 predicted
probabilities as long/short cutoffs. We freeze the same thresholds
here so the long-bucket and short-bucket definitions match the
backtest exactly — anyone reading this notebook gets the same
buckets that drove the headline Sharpe table.

In [2]:
val = base[base.fiscal_year == 2023].dropna(subset=feats + ['car_2_21'])
p_val = pipe.predict_proba(val[feats].astype(float).values)[:, 1]
short_cut, long_cut = quintile_thresholds(p_val, q=0.2)
print(f'val 2023 prob distribution:')
print(f'  20th pct (short cutoff): {short_cut:.4f}')
print(f'  80th pct (long cutoff):  {long_cut:.4f}')
print(f'  median:                  {np.median(p_val):.4f}')

val 2023 prob distribution:
  20th pct (short cutoff): 0.4203
  80th pct (long cutoff):  0.5729
  median:                  0.4978


## 3. Pooled OOS event-level tests

In [3]:
test_pool = base[base.fiscal_year.isin([2024, 2025])]
res_pool = event_level_tests(
    pipe, feats, test_pool,
    long_cut=long_cut, short_cut=short_cut,
    car_col='car_2_21',
)

print('=== Pooled OOS (test_2024 + test_2025) ===')
print(f'n events:           {res_pool.n_events}')
print(f'long bucket n:      {res_pool.long_n}')
print(f'short bucket n:     {res_pool.short_n}')
print()
print('--- Spearman ρ(prob, CAR_2_21) ---')
print(f'rho:                {res_pool.spearman_rho:+.4f}')
print(f'p (one-sided):      {res_pool.spearman_p_one_sided:.4f}')
print()
print('--- Mann-Whitney U (long > short) ---')
print(f'long mean CAR:      {res_pool.long_mean_car:+.4%}')
print(f'short mean CAR:     {res_pool.short_mean_car:+.4%}')
print(f'difference:         {res_pool.car_diff:+.4%}')
print(f'U statistic:        {res_pool.mw_u:.0f}')
print(f'p (one-sided):      {res_pool.mw_p_one_sided:.4f}')

=== Pooled OOS (test_2024 + test_2025) ===
n events:           363
long bucket n:      67
short bucket n:     87

--- Spearman ρ(prob, CAR_2_21) ---
rho:                +0.0907
p (one-sided):      0.0422

--- Mann-Whitney U (long > short) ---
long mean CAR:      +1.4310%
short mean CAR:     -0.2246%
difference:         +1.6556%
U statistic:        3434
p (one-sided):      0.0293


**Reading the numbers.** The README claim is "pooled p ≈ 0.07
(one-sided MW)". If our reproduction lands close to 0.07, the
README's claim is verified; if not, we need to investigate which
test (Spearman vs MW) the README actually meant and on which
exact subset.

If both p-values exceed 0.05 but cluster around 0.07–0.10, the
"weak positive but underpowered" framing of the paper's headline
is the correct read.

## 4. Per-year breakdown

Decompose the pooled result by year to see whether the signal is
consistent or driven by one year. Recall NB06 found the LLM block's
permutation Δ AUC was positive in 2024 (+1.77%) but negative in 2025
(−0.21%), so we expect the Spearman / MW signal to be stronger in
2024 too. If 2025 carries the pooled effect anyway, that would
indicate a tail-bucket phenomenon (extreme-prob events resolving
favourably even though aggregate ranking is noisy).

In [4]:
yearly = []
for year in [2024, 2025]:
    df_y = base[base.fiscal_year == year]
    res_y = event_level_tests(
        pipe, feats, df_y,
        long_cut=long_cut, short_cut=short_cut,
        car_col='car_2_21',
    )
    yearly.append({'year': year, **res_y.to_dict()})

yearly_df = pd.DataFrame(yearly)[
    ['year','n_events','long_n','short_n','spearman_rho','spearman_p_one_sided',
     'long_mean_car','short_mean_car','car_diff','mw_p_one_sided']
]
yearly_df.round(4)

,year,n_events,long_n,short_n,spearman_rho,spearman_p_one_sided,long_mean_car,short_mean_car,car_diff,mw_p_one_sided
0,2024,198,33,50,0.1099,0.0617,0.0123,-0.0099,0.0222,0.0585
1,2025,165,34,37,0.0730,0.1759,0.0163,0.0081,0.0082,0.2324


## 5. Outputs

In [5]:
out_dir = config.PROJECT_ROOT / 'output' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)

# Pooled result as a single-row table
pooled_df = pd.DataFrame([res_pool.to_dict()])
pooled_df.to_csv(out_dir / 'event_tests_pooled.csv', index=False)

# Per-year
yearly_df.to_csv(out_dir / 'event_tests_yearly.csv', index=False)

print('Saved:')
for f in sorted(out_dir.glob('event_tests_*.csv')):
    print(f'  {f.relative_to(config.PROJECT_ROOT)}')

Saved:
  output/tables/event_tests_pooled.csv
  output/tables/event_tests_yearly.csv
